# Pelatihan Model & Evaluasi: BiLSTM
Di sini kita memuat data preprocessed dan melatih model BiLSTM raksasa.

In [1]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Dropout, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)


I0000 00:00:1784217533.863821  103851 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784217533.920279  103851 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784217535.668640  103851 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Arsitektur BiLSTM

In [2]:
model = Sequential([
    Bidirectional(LSTM(64, activation='tanh'), input_shape=input_shape),
    Dropout(0.2),
    Dense(2)
], name="BiLSTM_Imdoukh")
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

E0000 00:00:1784217536.811266  103851 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/opt/jupyter-env/lib/python3.10/site-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "BiLSTM_Imdoukh"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        34,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,562 (135.01 KB)

 Trainable params: 34,562 (135.01 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_BiLSTM.csv', append=True)
EPOCHS = 200
BATCH_SIZE = 64

print(f"\n--- Training: BiLSTM ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)



--- Training: BiLSTM ---
Epoch 1/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0753 - mae: 0.2120 - val_loss: 0.0732 - val_mae: 0.2082
Epoch 2/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0723 - mae: 0.2078 - val_loss: 0.0713 - val_mae: 0.2007
Epoch 3/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0718 - mae: 0.2069 - val_loss: 0.0713 - val_mae: 0.1981
Epoch 4/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0716 - mae: 0.2066 - val_loss: 0.0703 - val_mae: 0.1993
Epoch 5/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0716 - mae: 0.2066 - val_loss: 0.0700 - val_mae: 0.1949
Epoch 6/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0715 - mae: 0.2066 - val_loss: 0.0710 - val_mae: 0.2009
Epoch 7/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0716 - mae: 0.2064 - val_loss: 0.0700 - val_mae: 0.1950
Epoch 8/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0715 - mae: 0.2065 - val_loss: 0.0710 - val_mae: 0.1971
Epoch 9/200
199/199 ━

## Evaluasi Kecepatan & Akurasi

In [4]:
single_sample = X_test[0:1]

# Pemanasan prediksi
_ = model.predict(single_sample, verbose=0)

# Precise inference benchmark
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Prediksi seluruh data test
pred = model.predict(X_test, verbose=0)
np.save('pred_BiLSTM.npy', pred)

results = {
    'BiLSTM': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)


,MSE,RMSE,MAE,R_Squared (R²),Speed (ms)
BiLSTM,0.074035,0.272093,0.209401,0.425842,62.197892
